# Amazon Nova Forge SDK - InspectLens Evaluation Quick Start

This notebook walks through running **InspectLens** evaluations using the Nova Forge SDK.
InspectLens is a job-based evaluation framework that runs benchmarks against your Nova model
via a SageMaker Training Job — no GPU required, since inference is delegated to a Bedrock
endpoint or an existing/new SageMaker endpoint.

## What You'll Learn

1. Setting up AWS credentials and S3 paths
2. Configuring `InspectLensConfig` for different inference providers
3. Creating benchmark tasks and running evaluations
4. Monitoring progress and retrieving results

## Table of Contents
- [Step 1: Import Required Modules](#step-1-import-required-modules)
- [Step 2: Configure Your AWS Resources](#step-2-configure-your-aws-resources)
- [Step 3: Configure Runtime Infrastructure](#step-3-configure-runtime-infrastructure)
- [Step 4: Initialize ForgeEvaluator](#step-4-initialize-forgeevaluator)
- [Step 5: Create Sample Benchmarks](#step-5-create-sample-benchmarks)
- [Step 6: Run InspectLens Evaluation](#step-6-run-inspectlens-evaluation)
  - [Option A: Bedrock Inference (default)](#option-a-bedrock-inference-default)
  - [Option B: Existing SageMaker Endpoint](#option-b-existing-sagemaker-endpoint)
  - [Option C: Create a New SageMaker Endpoint](#option-c-create-a-new-sagemaker-endpoint)
- [Step 7: Monitor and Retrieve Results](#step-7-monitor-and-retrieve-results)
- [Advanced: MLflow Tracking](#advanced-mlflow-tracking)
- [Summary](#summary)

## Prerequisites

- AWS credentials configured
- S3 bucket for evaluation artifacts and results
- IAM permissions for SageMaker and Bedrock
- Nova Forge SDK installed per [its README](https://github.com/aws/nova-forge-sdk/blob/main/README.md#installation)
- An `execution_role` with SageMaker and S3 permissions (required for InspectLens)

## Helpful Links
- See `docs/spec.md` for the full parameter reference.
- See `README.md` for a high-level overview of the Nova Forge SDK.

## Step 1: Import Required Modules

In [ ]:
!cd ../ && pip install .

In [ ]:
import os

import boto3
from botocore.exceptions import ClientError, NoCredentialsError, ProfileNotFound


def load_credentials(profile=None):
    """
    Load AWS credentials with fallback behavior.

    Args:
        profile (str, optional): AWS profile name. If provided, loads from credentials file.
                               If None, uses current authenticated AWS session.

    Returns:
        dict: Dictionary containing AWS credentials and region

    Raises:
        RuntimeError: If credential loading fails
    """
    if profile:
        # Try loading from credentials file
        try:
            session = boto3.Session(profile_name=profile)
            credentials = session.get_credentials()

            if not credentials:
                raise RuntimeError(f"No credentials found for profile '{profile}'")

        except ProfileNotFound:
            raise RuntimeError(f"Profile '{profile}' not found in credentials file")
        except Exception as e:
            raise RuntimeError(f"Failed to load credentials from file: {e}")

    else:
        # Try loading from current authenticated session
        try:
            session = boto3.Session()
            credentials = session.get_credentials()

            if not credentials:
                raise RuntimeError("No credentials found in current AWS session")

        except NoCredentialsError:
            raise RuntimeError("No AWS credentials configured")
        except Exception as e:
            raise RuntimeError(f"Failed to load credentials from current session: {e}")

        # Validate credentials by making a test call
    try:
        sts_client = session.client("sts")
        sts_client.get_caller_identity()
    except ClientError as e:
        raise RuntimeError(f"Invalid AWS credentials: {e}")
    except Exception as e:
        raise RuntimeError(f"Failed to validate credentials: {e}")

    return {
        "aws_access_key_id": credentials.access_key,
        "aws_secret_access_key": credentials.secret_key,
        "aws_session_token": credentials.token,
        "region_name": session.region_name or "us-east-1",
    }

In [ ]:
creds = load_credentials()

In [ ]:
from amzn_nova_forge import *
from amzn_nova_forge.core.enums import EvaluationTask, Model
from amzn_nova_forge.evaluator import ForgeEvaluator, InspectLensConfig

print("✅ SDK imported successfully!")

## Step 2: Configure Your AWS Resources

In [ ]:
# Replace the values below with your own S3 bucket and IAM execution role
S3_BUCKET = "your-bucket-name"  # Replace with your S3 bucket name, e.g. "my-nova-eval-bucket"
S3_OUTPUT_PATH = f"s3://{S3_BUCKET}/inspectlens/output"

# IAM role ARN with SageMaker + S3 permissions — required for InspectLens
# e.g. "arn:aws:iam::123456789012:role/YourSageMakerExecutionRole"
EXECUTION_ROLE = "arn:aws:iam::123456789012:role/YourSageMakerRole"

print(f"Output Path: {S3_OUTPUT_PATH}")
print(f"Execution Role: {EXECUTION_ROLE}")

## Step 3: Configure Runtime Infrastructure

InspectLens runs as a CPU-only SageMaker Training Job (the container acts as an orchestrator).
Use `SMTJRuntimeManager` with a CPU instance — no GPU needed.

In [ ]:
eval_infra = SMTJRuntimeManager(
    instance_type="ml.m5.large",  # CPU instance — no GPU required
    instance_count=1,
    execution_role=EXECUTION_ROLE,  # Required for InspectLens
)

print("✅ Runtime configured")
print(f"   Instance Type: {eval_infra.instance_type}")

## Step 4: Initialize ForgeEvaluator

In [ ]:
evaluator = ForgeEvaluator(
    model=Model.NOVA_LITE_2,
    infra=eval_infra,
    config=ForgeConfig(
        output_s3_path=S3_OUTPUT_PATH,
    ),
)

print("✅ ForgeEvaluator initialized — Model: Nova Lite 2.0")

## Step 5: Create Sample Benchmarks

InspectLens requires benchmark `.py` files with `@task` decorators.
The cell below creates a local `my_benchmarks/` directory with two ready-to-use tasks
that wrap built-in `inspect-evals` tasks.

**Important:** You must upload your benchmarks to S3 before starting an evaluation job.
Use `evaluator.upload_benchmarks(local_dir, s3_path)` to upload, then pass the returned
S3 URI as `benchmarks_path` in `InspectLensConfig`.

### Pre-installed packages

The InspectLens container comes with the following packages pre-installed:
`inspect-ai`, `boto3`, `aioboto3`, `openai`, `mlflow`, `pyyaml`.

Any additional dependencies your benchmarks require (e.g. `inspect-evals`) must be declared
in a `pyproject.toml` placed alongside your benchmark files. The container installs it automatically at job start.

In [ ]:
import os

BENCHMARKS_DIR = "./my_benchmarks"
os.makedirs(BENCHMARKS_DIR, exist_ok=True)

# BoolQ — reading comprehension (yes/no questions)
with open(f"{BENCHMARKS_DIR}/boolq_pt.py", "w") as f:
    f.write("""\
from inspect_ai import task
from inspect_evals.boolq import boolq


@task
def boolq_pt():
    return boolq()
""")

# MMLU Pro — multi-domain multiple choice
with open(f"{BENCHMARKS_DIR}/mmlu_pro_pt.py", "w") as f:
    f.write("""\
from inspect_ai import task
from inspect_evals.mmlu_pro import mmlu_pro


@task
def mmlu_pro_pt():
    return mmlu_pro()
""")

print(f"✅ Sample benchmarks created in {BENCHMARKS_DIR}/")
for f in os.listdir(BENCHMARKS_DIR):
    print(f"   {f}")

In [ ]:
# Create a pyproject.toml in the benchmarks directory so inspect-evals is installed
# when the SageMaker Training Job container starts up.
with open(f"{BENCHMARKS_DIR}/pyproject.toml", "w") as f:
    f.write("""\
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "my-benchmarks"
version = "0.1.0"
dependencies = [
    "inspect-evals",
]
""")

print(f"✅ pyproject.toml created in {BENCHMARKS_DIR}/")

In [ ]:
# Upload benchmarks to S3 — this is a separate step from starting the eval job.
# You choose where to store them and pass the S3 URI when configuring InspectLensConfig.
BENCHMARKS_S3_PATH = f"s3://{S3_BUCKET}/inspectlens/benchmarks/my_benchmarks/"

benchmarks_s3_uri = evaluator.upload_benchmarks(BENCHMARKS_DIR, BENCHMARKS_S3_PATH)
print(f"Benchmarks uploaded to: {benchmarks_s3_uri}")

## Step 6: Run InspectLens Evaluation

InspectLens supports three inference provider modes:
- **Bedrock** (default) — evaluates against a Bedrock model ID or ARN
- **Existing SageMaker endpoint** — evaluates against a deployed endpoint
- **New SageMaker endpoint** — creates an endpoint from model artifacts, evaluates, then optionally cleans up

Use `dry_run=True` first to validate your config locally without submitting a job.

### Option A: Bedrock Inference (default)

The simplest path — no endpoint management needed. Inference is routed through Bedrock.
If `bedrock_model_id` is omitted, the SDK uses the cross-region inference profile for
the `model` passed to `ForgeEvaluator`.

In [ ]:
inspect_config_bedrock = InspectLensConfig(
    benchmarks_path=benchmarks_s3_uri,  # S3 URI from upload_benchmarks()
    # bedrock_model_id="us.amazon.nova-2-lite-v1:0",  # defaults to ForgeEvaluator model
    tasks=[
        {"name": "boolq_pt", "limit": 100},
        {"name": "mmlu_pro_pt", "limit": 50},
    ],
    output_s3_path=f"{S3_OUTPUT_PATH}/bedrock-eval/",
)

# Validate config locally before submitting
evaluator.evaluate(
    job_name="inspectlens-bedrock-eval",
    eval_task=EvaluationTask.INSPECT_LENS,
    inspect_lens_config=inspect_config_bedrock,
    dry_run=True,
)
print("Config validated — ready to submit")

In [ ]:
eval_result_bedrock = evaluator.evaluate(
    job_name="inspectlens-bedrock-eval",
    eval_task=EvaluationTask.INSPECT_LENS,
    inspect_lens_config=inspect_config_bedrock,
    # overrides={"temperature": 0.0, "max_tokens": 4096, "max_connections": 8},
)

print("🚀 InspectLens evaluation started (Bedrock)!")
print(f"   Job ID: {eval_result_bedrock.job_id}")
print(f"   Results: {eval_result_bedrock.eval_output_path}")

### Option B: Existing SageMaker Endpoint

Use this when you already have a deployed SageMaker endpoint.

In [ ]:
# inspect_config_endpoint = InspectLensConfig(
#     benchmarks_path=benchmarks_s3_uri,  # S3 URI from upload_benchmarks()
#     endpoint_name="my-custom-nova-model-sagemaker",  # TODO: Replace
#     tasks=[{"name": "boolq_pt", "limit": 100}],
#     output_s3_path=f"{S3_OUTPUT_PATH}/endpoint-eval/",
# )
#
# eval_result_endpoint = evaluator.evaluate(
#     job_name="inspectlens-endpoint-eval",
#     eval_task=EvaluationTask.INSPECT_LENS,
#     inspect_lens_config=inspect_config_endpoint,
# )
# print(f"   Job ID: {eval_result_endpoint.job_id}")

### Option C: Create a New SageMaker Endpoint

Use this when you have model artifacts in S3 and want the SDK to spin up an endpoint,
run the evaluation, and optionally clean up the endpoint afterwards.

In [ ]:
# inspect_config_new_endpoint = InspectLensConfig(
#     benchmarks_path=benchmarks_s3_uri,  # S3 URI from upload_benchmarks()
#     model_s3_uri="s3://your-bucket/model-artifacts/",  # Replace with your model artifacts S3 path
#     inference_image_uri="123456789012.dkr.ecr.us-east-1.amazonaws.com/your-image:latest",  # Replace with your ECR image URI: <account>.dkr.ecr.<region>.amazonaws.com/<repo>:<tag>
#     endpoint_instance_type="ml.g5.12xlarge",
#     endpoint_execution_role_arn=EXECUTION_ROLE,
#     cleanup_endpoint=True,  # delete endpoint after eval (default: True)
#     tasks=[{"name": "boolq_pt", "limit": 100}],
#     output_s3_path=f"{S3_OUTPUT_PATH}/new-endpoint-eval/",
# )
#
# eval_result_new_ep = evaluator.evaluate(
#     job_name="inspectlens-new-endpoint-eval",
#     eval_task=EvaluationTask.INSPECT_LENS,
#     inspect_lens_config=inspect_config_new_endpoint,
# )
# print(f"   Job ID: {eval_result_new_ep.job_id}")

## Step 7: Monitor and Retrieve Results

In [ ]:
# Check job status
status, message = eval_result_bedrock.get_job_status()
print(f"Status: {status} — {message}")

In [ ]:
# Stream CloudWatch logs
evaluator.get_logs(job_result=eval_result_bedrock, limit=50, start_from_head=False)

In [ ]:
# Results are written as JSON logs to the output S3 path after the job completes
print(f"Results: {eval_result_bedrock.eval_output_path}")
print(f"Job ID:  {eval_result_bedrock.job_id}")

## Advanced: MLflow Tracking

Pass an `MLflowMonitor` to `ForgeConfig` and the SDK automatically populates the MLflow
tracking section in the InspectLens config.

Both tracking server ARNs (`mlflow-tracking-server/...`) and app ARNs (`mlflow-app/...`) are supported.

In [ ]:
# mlflow_monitor = MLflowMonitor(
#     tracking_uri="<Your MLflow App or Tracking Server ARN>",  # TODO: Replace
#     # arn:aws:sagemaker:<region>:<account>:mlflow-tracking-server/<name>
#     # arn:aws:sagemaker:<region>:<account>:mlflow-app/<name>
#     experiment_name="nova-inspectlens-evals",
#     run_name="inspectlens-run-1",
# )
#
# evaluator_with_mlflow = ForgeEvaluator(
#     model=Model.NOVA_LITE_2,
#     infra=eval_infra,
#     config=ForgeConfig(
#         output_s3_path=S3_OUTPUT_PATH,
#         mlflow_monitor=mlflow_monitor,  # SDK auto-populates MLflow in InspectLens config
#     ),
# )
#
# eval_result_mlflow = evaluator_with_mlflow.evaluate(
#     job_name="inspectlens-mlflow-eval",
#     eval_task=EvaluationTask.INSPECT_LENS,
#     inspect_lens_config=InspectLensConfig(
#         benchmarks_path=benchmarks_s3_uri,  # S3 URI from upload_benchmarks()
#         tasks=[{"name": "boolq_pt", "limit": 100}],
#         output_s3_path=f"{S3_OUTPUT_PATH}/mlflow-eval/",
#     ),
# )
# print(f"   Job ID: {eval_result_mlflow.job_id}")

## Summary

- `InspectLensConfig` controls benchmarks, inference provider, decoding, and output
- `benchmarks_path` must be an S3 URI — use `evaluator.upload_benchmarks(local_dir, s3_path)` to upload a local directory first
- Pass `eval_task=EvaluationTask.INSPECT_LENS` to `ForgeEvaluator.evaluate()`
- Use `dry_run=True` to validate config without submitting a job
- Pass `MLflowMonitor` via `ForgeConfig` to enable MLflow tracking

For more details, see `docs/spec.md` or the SDK README.